In [14]:
# 導入必要的套件
import paho.mqtt.client as mqtt
import time
import json

In [15]:
# MQTT Broker 設定（使用 HiveMQ 的公開測試 broker）
BROKER = "localhost"
PORT = 1883
TOPIC = "客廳/溫度"  # 您可以修改這個主題名稱

# 建立 MQTT 客戶端（使用最新的回調 API 版本 2）
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)

# 連線回調函數（API 版本 2 的簽名）
def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        print(f"✅ 成功連接到 MQTT Broker: {BROKER}")
    else:
        print(f"❌ 連線失敗，錯誤代碼: {reason_code}")

# 設定連線回調
client.on_connect = on_connect

# 連接到 Broker
print(f"正在連接到 {BROKER}...")
client.connect(BROKER, PORT, 60)
client.loop_start()  # 開始背景執行緒處理訊息

# 等待連線建立
time.sleep(1)


正在連接到 localhost...
✅ 成功連接到 MQTT Broker: localhost


In [16]:
# 發佈簡單的文字訊息
message = "Hello MQTT from Raspberry Pi!"
result = client.publish(TOPIC, message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"✅ 訊息已發佈到主題: {TOPIC}")
    print(f"   訊息內容: {message}")
else:
    print(f"❌ 發佈失敗，錯誤代碼: {result.rc}")


✅ 訊息已發佈到主題: 客廳/溫度
   訊息內容: Hello MQTT from Raspberry Pi!


In [17]:
# 發佈 JSON 格式的訊息
data = {
    "device": "Raspberry Pi",
    "temperature": 25.5,
    "humidity": 60,
    "timestamp": time.time()
}

json_message = json.dumps(data)
result = client.publish(TOPIC, json_message, qos=1)

if result.rc == mqtt.MQTT_ERR_SUCCESS:
    print(f"✅ JSON 訊息已發佈到主題: {TOPIC}")
    print(f"   訊息內容: {json_message}")
else:
    print(f"❌ 發佈失敗，錯誤代碼: {result.rc}")

✅ JSON 訊息已發佈到主題: 客廳/溫度
   訊息內容: {"device": "Raspberry Pi", "temperature": 25.5, "humidity": 60, "timestamp": 1764475019.2960513}


In [20]:
# 連續發佈多筆訊息（測試用）
print("開始連續發佈訊息...")
for i in range(5):
    message = f"測試訊息 #{i+1} - 時間: {time.strftime('%Y-%m-%d %H:%M:%S')}"
    result = client.publish(TOPIC, message, qos=1)
    
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"✅ [{i+1}/5] {message}")
    else:
        print(f"❌ [{i+1}/5] 發佈失敗")
    
    time.sleep(1)  # 等待 1 秒

print("\n✅ 所有訊息發佈完成！")

開始連續發佈訊息...
✅ [1/5] 測試訊息 #1 - 時間: 2025-11-30 11:58:34
✅ [2/5] 測試訊息 #2 - 時間: 2025-11-30 11:58:35
✅ [3/5] 測試訊息 #3 - 時間: 2025-11-30 11:58:36
✅ [4/5] 測試訊息 #4 - 時間: 2025-11-30 11:58:37
✅ [5/5] 測試訊息 #5 - 時間: 2025-11-30 11:58:38

✅ 所有訊息發佈完成！


In [21]:
# 關閉連線
client.loop_stop()
client.disconnect()
print("✅ MQTT 連線已關閉")

✅ MQTT 連線已關閉
